In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator

# visualization
import seaborn as sns
import matplotlib.pyplot as plt

from pyspark.ml.feature import VectorAssembler

In [2]:
LABEL_COLUMN='price'
SEED=42

spark = SparkSession.builder \
    .appName("Airbnb") \
    .config("spark.driver.memory", "24g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

raw_df = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("multiLine", "true")
    .option("quote", "\"")
    .option("escape", "\"")
    .option("mode", "PERMISSIVE")
    .parquet("../data/processed/airbnb_rio/total_data.parquet")
)

raw_df.createOrReplaceTempView('raw_listing')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/10 18:30:22 WARN Utils: Your hostname, rrghost, resolves to a loopback address: 127.0.1.1; using 192.168.15.3 instead (on interface enp8s0)
26/05/10 18:30:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 18:30:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# as the dataset is has a lot of columns, we did an analysis and selected the most 
# good appearing to the prediction. 

selected_df = spark.sql("""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        calculated_host_listings_count AS host_listings_count,
        property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification, 
        snapshot_month AS month,
        security_deposit,
        cleaning_fee
    FROM raw_listing
""")

selected_df.createOrReplaceTempView("selected_df")

## Initial Parsing

In [4]:
selected_df.show()

+-------------------+-------------------+-----------------+-------------------+-------------+------------+---------+--------+----+--------------------+------+-----------------------------+--------------------------------+-----+----------------+------------+
|           latitude|          longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|           amenities| price|require_guest_profile_picture|require_guest_phone_verification|month|security_deposit|cleaning_fee|
+-------------------+-------------------+-----------------+-------------------+-------------+------------+---------+--------+----+--------------------+------+-----------------------------+--------------------------------+-----+----------------+------------+
|          -22.99195|          -43.43592|              0.0|                2.0|        House|         2.0|      2.0|     1.0| 3.0|{TV,Wifi,"Air con...| 250.0|                          0.0|                             0.0|  8.0

Shape of the dataset:

In [5]:
selected_df.createOrReplaceTempView('selected_df')
row_count = spark.sql("SELECT COUNT(*) AS cnt FROM selected_df").first()['cnt']
print(f'Rows: {row_count}, Columns: {len(selected_df.columns)}')

Rows: 784121, Columns: 16


Checking for Null's:

In [6]:
selected_df.createOrReplaceTempView("selected_df")
null_count_expr = ",\n    ".join([
    f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
    for c in selected_df.columns
])
spark.sql(f"SELECT\n    {null_count_expr}\nFROM selected_df").show()

+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+----------------+------------+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|security_deposit|cleaning_fee|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+----------------+------------+
|       0|        0|              385|                  0|            0|           0|     1493|     775|2334|        0|    0|                            0|                               0|    0|          361062|      269335|
+--------+---------+-----------------+-------------------+-------------+------------+---------+-----

As we can see there's a lot of missing values in the `security_deposit` and `cleaning_fee` columns.

Dropping them from the database seems to be a good decision.

In [7]:
selected_df.createOrReplaceTempView('selected_df')
rows_before_drop = spark.sql("SELECT COUNT(*) AS cnt FROM selected_df").first()['cnt']
selected_df = spark.sql("""
    SELECT
        latitude, longitude, host_is_superhost, host_listings_count,
        property_type, accommodates, bathrooms, bedrooms, beds,
        amenities, price, require_guest_profile_picture,
        require_guest_phone_verification, month
    FROM selected_df
""")

In [8]:
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql("""
    SELECT * FROM selected_df
    WHERE latitude IS NOT NULL
      AND longitude IS NOT NULL
      AND host_is_superhost IS NOT NULL
      AND host_listings_count IS NOT NULL
      AND property_type IS NOT NULL
      AND accommodates IS NOT NULL
      AND bathrooms IS NOT NULL
      AND bedrooms IS NOT NULL
      AND beds IS NOT NULL
      AND amenities IS NOT NULL
      AND price IS NOT NULL
      AND require_guest_profile_picture IS NOT NULL
      AND require_guest_phone_verification IS NOT NULL
      AND month IS NOT NULL
""")
selected_df.createOrReplaceTempView('selected_df')
row_count = spark.sql("SELECT COUNT(*) AS cnt FROM selected_df").first()['cnt']
print(f"({row_count}, {len(selected_df.columns)}) - {rows_before_drop - row_count} rows dropped")

(780036, 14) - 4085 rows dropped


Checking if there is any missing left:

In [9]:
selected_df.createOrReplaceTempView("selected_df")
null_count_expr = ",\n    ".join([
    f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
    for c in selected_df.columns
])
spark.sql(f"SELECT\n    {null_count_expr}\nFROM selected_df").show()


+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|       0|        0|                0|                  0|            0|           0|        0|       0|   0|        0|    0|                            0|                               0|    0|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+



In [10]:
selected_df.createOrReplaceTempView('selected_df')

### Checking data types

In [11]:
selected_df.printSchema()

root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- host_is_superhost: double (nullable = true)
 |-- host_listings_count: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- accommodates: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- beds: double (nullable = true)
 |-- amenities: string (nullable = true)
 |-- price: double (nullable = true)
 |-- require_guest_profile_picture: double (nullable = true)
 |-- require_guest_phone_verification: double (nullable = true)
 |-- month: double (nullable = true)



TODO: A brief explanation of what need to be changed

### Checking data types

In [12]:
selected_df = spark.sql("""
    SELECT
        TRY_CAST(latitude AS DOUBLE) AS latitude,
        TRY_CAST(longitude AS DOUBLE) AS longitude,
        TRY_CAST(host_is_superhost AS BOOLEAN) AS host_is_superhost,
        TRY_CAST(TRY_CAST(host_listings_count AS DOUBLE) AS INT) AS host_listings_count,
        property_type,
        TRY_CAST(TRY_CAST(accommodates AS DOUBLE) AS INT) AS accommodates,
        TRY_CAST(TRY_CAST(bathrooms AS DOUBLE) AS INT) AS bathrooms,
        TRY_CAST(TRY_CAST(bedrooms AS DOUBLE) AS INT) AS bedrooms,
        TRY_CAST(TRY_CAST(beds AS DOUBLE) AS INT) AS beds,
        amenities,
        TRY_CAST(REGEXP_REPLACE(price, '[\\\\$,]', '') AS DOUBLE) AS price,
        TRY_CAST(require_guest_profile_picture AS BOOLEAN) AS require_guest_profile_picture,
        TRY_CAST(require_guest_phone_verification AS BOOLEAN) AS require_guest_phone_verification, 
        TRY_CAST(month AS INT) AS month
    FROM selected_df
""")

selected_df.createOrReplaceTempView("selected_df")

null_count_expr = ",\n    ".join([
    f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
    for c in selected_df.columns
])
spark.sql(f"SELECT\n    {null_count_expr}\nFROM selected_df").show()

+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|       0|        0|                0|                  0|            0|           0|        0|       0|   0|        0|    0|                            0|                               0|    0|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+



## EDA + Data Cleaning

### Dealing with outliers

In [13]:
def get_max_fence(column):
    row = spark.sql(f"""
        SELECT
            PERCENTILE_APPROX(`{column}`, 0.25) AS q1,
            PERCENTILE_APPROX(`{column}`, 0.75) AS q3
        FROM selected_df
    """).first()
    q1, q3 = row['q1'], row['q3']
    iqr = q3 - q1
    return q3 + 1.5 * iqr

#### `host_listing_count`

In [14]:
listing_count_max_fence = get_max_fence('host_listings_count')
print(listing_count_max_fence)

6.0


In [15]:
selected_df.createOrReplaceTempView('selected_df')
rows_before = spark.sql("SELECT COUNT(*) AS cnt FROM selected_df").first()['cnt']
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE host_listings_count <= {listing_count_max_fence}
""")
selected_df.createOrReplaceTempView('selected_df')
rows_after = spark.sql("SELECT COUNT(*) AS cnt FROM selected_df").first()['cnt']
print(f"{rows_before - rows_after} rows were deleted.")

91605 rows were deleted.


In [16]:
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql("""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        CASE WHEN host_listings_count = 0.0 THEN 1.0 ELSE host_listings_count END AS host_listings_count,
        property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month
    FROM selected_df
""")

#### `price`

In [17]:
price_max_fence = get_max_fence('price')
print(price_max_fence)

1276.0


In [18]:
selected_df.createOrReplaceTempView('selected_df')
rows_before = spark.sql("SELECT COUNT(*) AS cnt FROM selected_df").first()['cnt']
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE price <= {price_max_fence}
""")
selected_df.createOrReplaceTempView('selected_df')
rows_after = spark.sql("SELECT COUNT(*) AS cnt FROM selected_df").first()['cnt']
print(f"{rows_before - rows_after} rows were deleted.")

66859 rows were deleted.


#### `property_type`

In [19]:
categories_to_append = ('Aparthotel', 'Earth house', 'Chalet', 'Cottage', 'Tiny house',
                        'Boutique hotel', 'Hotel', 'Casa particular (Cuba)', 'Bungalow',
                        'Nature lodge', 'Cabin', 'Castle', 'Treehouse', 'Island', 'Boat', 'Tent',
                        'Resort', 'Hut', 'Campsite', 'Barn', 'Dorm', 'Camper/RV', 'Farm stay', 'Yurt',
                        'Tipi', 'Pension (South Korea)', 'Dome house', 'Igloo', 'Casa particular',
                        'Houseboat', 'Lighthouse', 'Plane', 'Train', 'Parking Space')

category_list_sql = ", ".join([f"'" + c.replace("'", "''") + "'" for c in categories_to_append])
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        CASE WHEN property_type IN ({category_list_sql}) THEN 'Other' ELSE property_type END AS property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month
    FROM selected_df
""")

#### `beds`

In [20]:
beds_max_fence = get_max_fence('beds')
print(beds_max_fence)

6.0


In [21]:
selected_df.createOrReplaceTempView('selected_df')
rows_before = spark.sql("SELECT COUNT(*) AS cnt FROM selected_df").first()['cnt']
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE beds <= {beds_max_fence}
""")
selected_df.createOrReplaceTempView('selected_df')
rows_after = spark.sql("SELECT COUNT(*) AS cnt FROM selected_df").first()['cnt']
print(f"{rows_before - rows_after} rows were deleted.")

14147 rows were deleted.


#### `amenities`

In [22]:
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql("""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month,
        SIZE(SPLIT(amenities, ',')) + 1 AS n_amenities
    FROM selected_df
""")
selected_df.createOrReplaceTempView('selected_df')
mode_value = spark.sql("""
    SELECT n_amenities
    FROM selected_df
    WHERE amenities <> '{}'
    GROUP BY n_amenities
    ORDER BY COUNT(*) DESC
    LIMIT 1
""").first()['n_amenities']
selected_df = spark.sql(f"""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month,
        CASE WHEN amenities = '{{}}' THEN {mode_value} ELSE n_amenities END AS n_amenities
    FROM selected_df
""")

In [23]:
amenities_max_fence = get_max_fence('n_amenities')
print(amenities_max_fence)

42.0


In [24]:
selected_df.createOrReplaceTempView('selected_df')
rows_before = spark.sql("SELECT COUNT(*) AS cnt FROM selected_df").first()['cnt']
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE n_amenities <= {amenities_max_fence}
""")
selected_df.createOrReplaceTempView('selected_df')
rows_after = spark.sql("SELECT COUNT(*) AS cnt FROM selected_df").first()['cnt']
print(f"{rows_before - rows_after} rows were deleted.")

17991 rows were deleted.


## Encoding

In [25]:
selected_df.createOrReplaceTempView('selected_df')
df_label_encoder = spark.sql("""
    WITH property_type_index AS (
        SELECT
            property_type,
            ROW_NUMBER() OVER (ORDER BY cnt DESC, property_type ASC) - 1 AS property_type_encoded
        FROM (
            SELECT property_type, COUNT(*) AS cnt
            FROM selected_df
            GROUP BY property_type
        )
    )
    SELECT
        s.latitude,
        s.longitude,
        CASE
            WHEN s.host_is_superhost IS TRUE THEN 1
            WHEN s.host_is_superhost IS FALSE THEN 0
            ELSE CAST(s.host_is_superhost AS INT)
        END AS host_is_superhost,
        s.host_listings_count,
        p.property_type_encoded AS property_type,
        s.accommodates,
        s.bathrooms,
        s.bedrooms,
        s.beds,
        s.price,
        CASE
            WHEN s.require_guest_profile_picture IS TRUE THEN 1
            WHEN s.require_guest_profile_picture IS FALSE THEN 0
            ELSE CAST(s.require_guest_profile_picture AS INT)
        END AS require_guest_profile_picture,
        CASE
            WHEN s.require_guest_phone_verification IS TRUE THEN 1
            WHEN s.require_guest_phone_verification IS FALSE THEN 0
            ELSE CAST(s.require_guest_phone_verification AS INT)
        END AS require_guest_phone_verification,
        s.month,
        s.n_amenities
    FROM selected_df s
    LEFT JOIN property_type_index p
        ON s.property_type = p.property_type
""")
print('Columns encoded')

Columns encoded


In [26]:
df_label_encoder.createOrReplaceTempView("df_label_encoder")
df_label_encoder = spark.sql("""
    SELECT *,
        CAST(
            CASE
                WHEN price < 200  THEN 0
                WHEN price < 500  THEN 1
                ELSE 2
            END
        AS DOUBLE) AS price_band
    FROM df_label_encoder
""")
df_label_encoder.createOrReplaceTempView("df_label_encoder")
print("Band distribution (0=budget<200, 1=mid 200-499, 2=premium>=500):")
spark.sql("""
    SELECT price_band, COUNT(*) AS cnt,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM df_label_encoder
    GROUP BY price_band
    ORDER BY price_band
""").show()

Band distribution (0=budget<200, 1=mid 200-499, 2=premium>=500):


+----------+------+----+
|price_band|   cnt| pct|
+----------+------+----+
|       0.0|235245|39.9|
|       1.0|213811|36.3|
|       2.0|140378|23.8|
+----------+------+----+



## Model

In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor
models = [
    (
        "Linear Regression",
        LinearRegression(
            labelCol=LABEL_COLUMN,
            featuresCol="features",
            maxIter=200,
            regParam=0.01,
            elasticNetParam=0.0
        )
    ),
    (
        "Decision Tree Regressor",
        DecisionTreeRegressor(
            labelCol=LABEL_COLUMN,
            featuresCol="features",
            maxDepth=30,
            minInstancesPerNode=15,
            minInfoGain=0.0,
            maxBins=32,
            seed=SEED
        )
    ),
]

In [28]:
feature_cols = [
    "host_is_superhost",
    "host_listings_count",
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "month",
    "property_type",
    "require_guest_profile_picture",
    "require_guest_phone_verification",
    "n_amenities"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
prepared_df = assembler.transform(df_label_encoder)

train_prepared, test_prepared = prepared_df.randomSplit([0.8, 0.2], seed=SEED)

train_prepared.createOrReplaceTempView('train_prepared')
test_prepared.createOrReplaceTempView('test_prepared')
train_count = spark.sql("SELECT COUNT(*) AS cnt FROM train_prepared").first()['cnt']
test_count = spark.sql("SELECT COUNT(*) AS cnt FROM test_prepared").first()['cnt']
print(f"Linhas no Treino: {train_count}")
print(f"Linhas no Teste: {test_count}")

Linhas no Treino: 471762
Linhas no Teste: 117672


In [29]:
def evaluate_model(model_name: str, predictions) -> None:
    predictions = predictions.cache()
    predictions.createOrReplaceTempView("predictions")

    rmse = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="rmse",
    ).evaluate(predictions)
    mae = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="mae",
    ).evaluate(predictions)
    r2 = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="r2",
    ).evaluate(predictions)

    print(f"\n{model_name}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R2: {r2:.6f}")

    predictions.unpersist()

In [30]:
for model_name, estimator in models:
    model = estimator.fit(train_prepared)
    predictions = model.transform(test_prepared)
    evaluate_model(model_name, predictions)


Linear Regression
RMSE: 240.64
MAE: 179.97
R2: 0.292643


26/05/10 18:31:14 ERROR Executor: Exception in task 13.0 in stage 187.0 (TID 1547)
java.io.StreamCorruptedException: invalid handle value: 00000000
	at java.base/java.io.ObjectInputStream.readHandle(ObjectInputStream.java:1842)
	at java.base/java.io.ObjectInputStream.readClassDesc(ObjectInputStream.java:1900)
	at java.base/java.io.ObjectInputStream.readArray(ObjectInputStream.java:2112)
	at java.base/java.io.ObjectInputStream.readObject0(ObjectInputStream.java:1721)
	at java.base/java.io.ObjectInputStream$FieldValues.<init>(ObjectInputStream.java:2606)
	at java.base/java.io.ObjectInputStream.readSerialData(ObjectInputStream.java:2457)
	at java.base/java.io.ObjectInputStream.readOrdinaryObject(ObjectInputStream.java:2257)
	at java.base/java.io.ObjectInputStream.readObject0(ObjectInputStream.java:1733)
	at java.base/java.io.ObjectInputStream$FieldValues.<init>(ObjectInputStream.java:2606)
	at java.base/java.io.ObjectInputStream.readSerialData(ObjectInputStream.java:2457)
	at java.base/ja

Py4JJavaError: An error occurred while calling o179.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 13 in stage 187.0 failed 1 times, most recent failure: Lost task 13.0 in stage 187.0 (TID 1547) (192.168.15.3 executor driver): java.io.StreamCorruptedException: invalid handle value: 00000000
	at java.base/java.io.ObjectInputStream.readHandle(ObjectInputStream.java:1842)
	at java.base/java.io.ObjectInputStream.readClassDesc(ObjectInputStream.java:1900)
	at java.base/java.io.ObjectInputStream.readArray(ObjectInputStream.java:2112)
	at java.base/java.io.ObjectInputStream.readObject0(ObjectInputStream.java:1721)
	at java.base/java.io.ObjectInputStream$FieldValues.<init>(ObjectInputStream.java:2606)
	at java.base/java.io.ObjectInputStream.readSerialData(ObjectInputStream.java:2457)
	at java.base/java.io.ObjectInputStream.readOrdinaryObject(ObjectInputStream.java:2257)
	at java.base/java.io.ObjectInputStream.readObject0(ObjectInputStream.java:1733)
	at java.bas

## MLP Classifier (Price Bands)

In [ ]:
mlp_assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
mlp_df = mlp_assembler.transform(df_label_encoder)
train_mlp, test_mlp = mlp_df.randomSplit([0.8, 0.2], seed=SEED)

# layers: 13 inputs → 64 → 32 → 3 price bands
mlp = MultilayerPerceptronClassifier(
    labelCol="price_band",
    featuresCol="features",
    layers=[13, 64, 32, 3],
    maxIter=200,
    blockSize=128,
    solver="l-bfgs",
    tol=1e-6,
    seed=SEED
)

mlp_model = mlp.fit(train_mlp)
mlp_predictions = mlp_model.transform(test_mlp).cache()
mlp_predictions.createOrReplaceTempView("mlp_predictions")

accuracy = MulticlassClassificationEvaluator(
    labelCol="price_band", predictionCol="prediction", metricName="accuracy"
).evaluate(mlp_predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol="price_band", predictionCol="prediction", metricName="f1"
).evaluate(mlp_predictions)

print("\nMLP Classifier (Price Bands: 0=<200, 1=200-499, 2=>=500)")
print(f"Accuracy: {accuracy:.4f}")
print(f"Weighted F1: {f1:.4f}")

print("\nPer-band accuracy:")
spark.sql("""
    SELECT price_band,
           SUM(CASE WHEN price_band = prediction THEN 1 ELSE 0 END) AS correct,
           COUNT(*) AS total,
           ROUND(SUM(CASE WHEN price_band = prediction THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS acc_pct
    FROM mlp_predictions
    GROUP BY price_band
    ORDER BY price_band
""").show()

mlp_predictions.unpersist()